# SpaceX Falcon 9 First Stage Landing Prediction

## Lab 5: Exploratory Data Analysis with SQL

In this lab, we load the SpaceX dataset into a SQLite database and explore it using SQL queries. Understanding the data — launch sites, payload masses, booster versions, mission outcomes and landing outcomes — is a critical step before building predictive models.

## Overview of the dataset

The dataset `dataset_part_2.csv` contains one row per Falcon 9 launch with the columns
`FlightNumber`, `Date`, `BoosterVersion`, `PayloadMass`, `Orbit`, `LaunchSite`, `Outcome`,
`Flights`, `GridFins`, `Reused`, `Legs`, `LandingPad`, `Block`, `ReusedCount`, `Serial`,
`Longitude`, `Latitude` and the binary target `Class`.

`Outcome` records how the first stage ended: `True RTLS` (return-to-launch-site pad),
`True ASDS` (autonomous drone ship), `True Ocean` (controlled splashdown), the matching
`False *` failures, or `None` when no attempt was made. `Class` is 1 for `True RTLS` /
`True ASDS` and 0 otherwise.

We load the table into SQLite as `SPACEXTBL` and query it with the `ipython-sql` magic.

In [ ]:
!pip install ipython-sql -q

In [ ]:
%load_ext sql

import sqlite3
import pandas as pd

con = sqlite3.connect("my_data1.db")
cur = con.cursor()

DATA_URL = ("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
            "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv")

In [ ]:
df = pd.read_csv(DATA_URL)
df.to_sql("SPACEXTBL", con, if_exists="replace", index=False, method="multi")
print("Loaded", df.shape[0], "rows and", df.shape[1], "columns into SPACEXTBL")
df.head()

In [ ]:
%sql sqlite:///my_data1.db

### Task 1: Display the names of the unique launch sites in the space mission

In [ ]:
%sql SELECT DISTINCT LaunchSite FROM SPACEXTBL

### Task 2: Display 5 records where launch sites begin with the string 'CCA'

The `LIKE 'CCA%'` pattern matches every launch from Cape Canaveral (CCAFS SLC 40).

In [ ]:
%sql SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' LIMIT 5

### Task 3: Display the total payload mass carried from CCAFS SLC 40

In [ ]:
%sql SELECT SUM(PayloadMass) AS Total_Payload_Mass FROM SPACEXTBL WHERE LaunchSite = 'CCAFS SLC 40'

### Task 4: Display the average payload mass carried by Block 5 boosters

In [ ]:
%sql SELECT AVG(PayloadMass) AS Avg_Payload_Mass FROM SPACEXTBL WHERE Block = 5

### Task 5: List the date of the first successful landing on a ground pad

We take `MIN(Date)` over rows whose `Outcome` is `True RTLS`, i.e. a return-to-launch-site pad landing.

In [ ]:
%sql SELECT MIN(Date) AS First_Successful_Ground_Pad_Landing FROM SPACEXTBL WHERE Outcome = 'True RTLS'

### Task 6: List the booster blocks with a successful drone-ship landing and a payload between 4,000 and 6,000 kg

In [ ]:
%sql SELECT DISTINCT Block FROM SPACEXTBL WHERE Outcome = 'True ASDS' AND PayloadMass > 4000 AND PayloadMass < 6000

### Task 7: Count the landing outcomes

In [ ]:
%sql SELECT Outcome, COUNT(*) AS Total FROM SPACEXTBL GROUP BY Outcome ORDER BY Total DESC

### Task 8: List the booster blocks that carried the maximum payload mass

A subquery returns the maximum payload mass in the table, then we select the blocks that match it.

In [ ]:
%sql SELECT DISTINCT Block FROM SPACEXTBL WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL)

### Task 9: List the failed drone-ship landings in 2015 with their booster block and launch site

`Date` is stored as text in `YYYY-MM-DD` format, so `substr(Date, 6, 2)` extracts the month
and `substr(Date, 1, 4)` extracts the year.

In [ ]:
%sql SELECT substr(Date, 6, 2) AS Month, Outcome, Block, LaunchSite FROM SPACEXTBL WHERE Outcome = 'False ASDS' AND substr(Date, 1, 4) = '2015'

### Task 10: Rank the count of landing outcomes between 2010-06-04 and 2017-03-20

We group the outcomes inside that window and order them by count, descending.

In [ ]:
%sql SELECT Outcome, COUNT(*) AS Outcome_Count FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Outcome ORDER BY Outcome_Count DESC

In [ ]:
con.close()

## Summary

SQL confirms the shape of the problem: flights launch from three pads (CCAFS SLC 40 carries
the bulk of them); CCAFS SLC 40 alone accounts for 305,151 kg of payload; Block 5 boosters
average far heavier payloads than the early blocks; the first successful ground-pad landing
was on 2015-12-22; successful drone-ship landings dominate the outcome counts; the heaviest
payloads flew on Block 5; and every 2015 drone-ship attempt failed, which is exactly the
early-era pattern seen in the visual EDA. The last query also yields the 60 / 30 class split
that the machine-learning notebook has to beat.